In [12]:
import numpy as np
import pandas as pd

# For reproducible results
np.random.seed(42)

In [13]:
n_samples = 2000

data = {
    "Sleep_Hours": np.random.uniform(3, 10, n_samples),

    "Previous_Screen_Time": np.random.uniform(0, 12, n_samples),

    "Stress_Level": np.random.randint(1, 11, n_samples),

    "Caffeine_Intake": np.random.randint(0, 6, n_samples),

    "Study_Time": np.random.choice(
        ["Morning", "Afternoon", "Evening", "Night"],
        n_samples
    ),

    "Breaks_Taken": np.random.randint(0, 7, n_samples),

    "Task_Difficulty": np.random.randint(1, 11, n_samples),

    "Previous_Session_Score": np.random.uniform(0, 100, n_samples),

    "Phone_Notifications": np.random.randint(0, 21, n_samples),

    "Environment_Noise": np.random.randint(0, 11, n_samples),

    "Physical_Activity": np.random.randint(0, 121, n_samples),

    "Hydration_Level": np.random.randint(1, 11, n_samples),

    "Study_Environment": np.random.choice(
        ["Home", "Library", "Cafe"],
        n_samples
    )
}

df = pd.DataFrame(data)

df.head(10)

,Sleep_Hours,Previous_Screen_Time,Stress_Level,Caffeine_Intake,Study_Time,Breaks_Taken,Task_Difficulty,Previous_Session_Score,Phone_Notifications,Environment_Noise,Physical_Activity,Hydration_Level,Study_Environment
0,5.621781,3.140468,10,1,Morning,0,2,47.028848,17,10,40,2,Home
1,9.655000,2.963746,5,1,Afternoon,6,6,45.880695,17,4,92,2,Library
2,8.123958,10.875055,5,5,Evening,6,8,98.364253,12,9,69,2,Library
3,7.190609,2.994554,7,3,Morning,3,7,39.244485,7,7,81,10,Cafe
4,4.092130,3.263397,4,4,Afternoon,6,1,36.229543,11,8,48,7,Cafe
5,4.091962,9.112779,4,1,Afternoon,2,5,31.860437,20,7,43,6,Cafe
6,3.406585,5.396878,9,4,Morning,5,2,89.474922,0,2,34,1,Cafe
7,9.063233,9.320527,6,0,Night,2,6,49.672383,19,8,90,2,Home
8,7.207805,0.784394,4,1,Morning,2,7,60.001448,17,2,4,1,Cafe
9,7.956508,5.850854,2,1,Night,6,7,18.081848,2,7,70,7,Cafe


In [14]:
# Generate Focus Score
focus_score = (
    30
    + df["Sleep_Hours"] * 4
    + df["Previous_Session_Score"] * 0.15
    + df["Hydration_Level"] * 1.2
    + df["Physical_Activity"] * 0.05
    + df["Breaks_Taken"] * 1.5
    - df["Previous_Screen_Time"] * 1.2
    - df["Stress_Level"] * 1.8
    - df["Phone_Notifications"] * 0.7
    - df["Environment_Noise"] * 0.8
    - df["Task_Difficulty"] * 0.8
    + np.random.normal(0, 5, n_samples)
)

# Add time and environment effects
focus_score += df["Study_Time"].map({
    "Morning": 5,
    "Afternoon": 0,
    "Evening": 2,
    "Night": -3
})

focus_score += df["Study_Environment"].map({
    "Home": 0,
    "Library": 4,
    "Cafe": -2
})

df["Focus_Score"] = focus_score.clip(0, 100).round(2)

# Generate Classification Target
success_probability = 1 / (
    1 + np.exp(-(df["Focus_Score"] - 60) / 8)
)

df["Session_Success"] = np.random.binomial(
    1, success_probability
)

# Save dataset
from pathlib import Path

Path("../data").mkdir(parents=True, exist_ok=True)
df.to_csv("../data/focusdna_dataset.csv", index=False)

df.head(20)

,Sleep_Hours,Previous_Screen_Time,Stress_Level,Caffeine_Intake,Study_Time,Breaks_Taken,Task_Difficulty,Previous_Session_Score,Phone_Notifications,Environment_Noise,Physical_Activity,Hydration_Level,Study_Environment,Focus_Score,Session_Success
0,5.621781,3.140468,10,1,Morning,0,2,47.028848,17,10,40,2,Home,33.29,0
1,9.655000,2.963746,5,1,Afternoon,6,6,45.880695,17,4,92,2,Library,58.29,1
2,8.123958,10.875055,5,5,Evening,6,8,98.364253,12,9,69,2,Library,51.15,0
3,7.190609,2.994554,7,3,Morning,3,7,39.244485,7,7,81,10,Cafe,56.54,0
4,4.092130,3.263397,4,4,Afternoon,6,1,36.229543,11,8,48,7,Cafe,50.38,0
5,4.091962,9.112779,4,1,Afternoon,2,5,31.860437,20,7,43,6,Cafe,16.19,0
6,3.406585,5.396878,9,4,Morning,5,2,89.474922,0,2,34,1,Cafe,46.45,0
7,9.063233,9.320527,6,0,Night,2,6,49.672383,19,8,90,2,Home,28.30,0
8,7.207805,0.784394,4,1,Morning,2,7,60.001448,17,2,4,1,Cafe,52.57,0
9,7.956508,5.850854,2,1,Night,6,7,18.081848,2,7,70,7,Cafe,53.96,0


In [15]:
print("Dataset Shape:", df.shape)
print("\nMissing Values:\n", df.isnull().sum())
print("\nTarget Distribution:\n", df["Session_Success"].value_counts())
print("\nFocus Score Statistics:\n", df["Focus_Score"].describe())

Dataset Shape: (2000, 15)

Missing Values:
 Sleep_Hours               0
Previous_Screen_Time      0
Stress_Level              0
Caffeine_Intake           0
Study_Time                0
Breaks_Taken              0
Task_Difficulty           0
Previous_Session_Score    0
Phone_Notifications       0
Environment_Noise         0
Physical_Activity         0
Hydration_Level           0
Study_Environment         0
Focus_Score               0
Session_Success           0
dtype: int64

Target Distribution:
 Session_Success
0    1481
1     519
Name: count, dtype: int64

Focus Score Statistics:
 count    2000.000000
mean       46.691365
std        15.063254
min         0.000000
25%        36.500000
50%        46.685000
75%        57.447500
max        90.300000
Name: Focus_Score, dtype: float64
